# P-value cutoff sweep

For each dataset, sweep the p-value cutoff and count the number of targets crossing it, split into **targeting** (blue) and **NC** (orange). Right panel: % NCs among the hits — a per-cutoff false-positive rate from the controls.

NCs here stand in for the OR-gene class on the original benchmark slide (production libraries don't carry an OR class).

**Input:** `data/<dataset>/energy_distance/wg1_significant_tfs.tsv` (per-dataset, from the jamboree data dir)
**Output:** `results/pval_cutoff_sweep/cutoff_sweep_<short>.pdf`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

HERE = Path.cwd()
EDIST = HERE if HERE.name == "edist" else HERE.parent
JAMB = EDIST.parents[2]
REPO = JAMB.parents[2]
RESULTS = EDIST / "results" / "pval_cutoff_sweep"
RESULTS.mkdir(parents=True, exist_ok=True)

with open(REPO / "config/colors/production_TF-Perturb-seq.yaml") as f:
    CFG = yaml.safe_load(f)

SHORT = {
    "Hon_WTC11-cardiomyocyte-differentiation_TF-Perturb-seq": "HonCM",
    "Huangfu_HUES8-definitive-endoderm-differentiation_TF-Perturb-seq": "HuangfuDE",
    "Huangfu_HUES8-embryonic-stemcell-differentiation_TF-Perturb-seq": "HuangfuESC",
    "Gersbach_WTC11-hepatocyte-differentiation_TF-Perturb-seq": "GersbachHep",
    "Engreitz_WTC11-endothelial-cells_TF-Perturb-seq": "EngreitzEndo",
}
SHORT_TO_FULL = {v: k for k, v in SHORT.items()}
DATASET_ORDER = ["HonCM", "HuangfuDE", "HuangfuESC", "GersbachHep"]
COLORS = {s: CFG["dataset_colors"][SHORT_TO_FULL[s]] for s in DATASET_ORDER}

print("RESULTS:", RESULTS)


In [ ]:
CUTOFFS = [1.0, 0.1, 0.05, 0.01, 0.005, 0.001]

def cutoff_sweep_panel(df: pd.DataFrame, title: str, ax1, ax2) -> None:
    is_nc = df["type"] == "negative control"
    n_total = [(df["pval_mean"] < c).sum() for c in CUTOFFS]
    n_nc = [((df["pval_mean"] < c) & is_nc).sum() for c in CUTOFFS]
    n_target = [t - nc for t, nc in zip(n_total, n_nc)]
    pct_nc = [100 * nc / t if t else 0 for nc, t in zip(n_nc, n_total)]
    x = np.arange(len(CUTOFFS))

    ax1.bar(x, n_target, color="#3B6FB6", label="Targeting")
    ax1.bar(x, n_nc, bottom=n_target, color="#E07B30", label="NC")
    for xi, t in zip(x, n_total):
        ax1.text(xi, t, f"{t}", ha="center", va="bottom", fontsize=8)
    ax1.set_xticks(x); ax1.set_xticklabels([str(c) for c in CUTOFFS])
    ax1.set_xlabel("p-value cutoff"); ax1.set_ylabel("# Targets")
    ax1.set_title(f"{title}  a", loc="left", fontweight="bold")
    ax1.legend(frameon=False, fontsize=8)
    ax1.spines[["top", "right"]].set_visible(False)

    ax2.bar(x, pct_nc, color="#3B6FB6")
    for xi, p in zip(x, pct_nc):
        ax2.text(xi, p, f"{p:.1f}", ha="center", va="bottom", fontsize=8)
    ax2.set_xticks(x); ax2.set_xticklabels([str(c) for c in CUTOFFS])
    ax2.set_xlabel("p-value cutoff"); ax2.set_ylabel("% NCs among hits")
    ax2.set_title("b", loc="left", fontweight="bold")
    ax2.spines[["top", "right"]].set_visible(False)
    ax2.set_ylim(0, max(max(pct_nc) * 1.18, 5))

for short in DATASET_ORDER:
    full = SHORT_TO_FULL[short]
    p = JAMB / "data" / full / "energy_distance" / "wg1_significant_tfs.tsv"
    if not p.is_file():
        print(f"skip {short}: {p} not found")
        continue
    df = pd.read_csv(p, sep="\t")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
    cutoff_sweep_panel(df, short, ax1, ax2)
    fig.suptitle(short)
    plt.tight_layout()
    out = RESULTS / f"cutoff_sweep_{short}.pdf"
    fig.savefig(out)
    plt.show()
    print(f"wrote {out}")